# Bagging Regressor - Predição de Gastos de Saúde de Funcionários

In [546]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import ttest_ind
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import chi2


## Ler dados
- Não há dados nulos, logo não há um tratamento para valores ausentes em primeira análise.

In [547]:
df_costs = pd.read_csv("./dataset/employees.csv")
df_costs.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              1338 non-null   int64  
 1   sex              1338 non-null   str    
 2   bmi              1338 non-null   float64
 3   children         1338 non-null   int64  
 4   smoker           1338 non-null   str    
 5   region           1338 non-null   str    
 6   medical charges  1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [548]:
df_costs.head()

,age,sex,bmi,children,smoker,region,medical charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Feature Engineering - Variáveis Categóricas

### Checar se alguma coluna tem somente um único valor possível
- Nenhuma coluna tem somente um valor possível.

In [549]:
for column in df_costs.select_dtypes(include='str').columns.tolist(): # type: ignore
    if df_costs[column].nunique() == 1:
        print(column)

### Mostrar os valores possíveis para cada coluna categórica
- Realmente não há valores nulos para as variáveis categóricas.

In [550]:
for column in df_costs.select_dtypes(include='str').columns.tolist(): # type: ignore
    print(f"Valores na coluna {column} = {list(df_costs[column].unique())}")

Valores na coluna sex = ['female', 'male']
Valores na coluna smoker = ['yes', 'no']
Valores na coluna region = ['southwest', 'southeast', 'northwest', 'northeast']


## Feature Engineering - Variáveis Numéricas

### Olhar dados específicos de distribuição das variáveis
- As variáveis assumem valores reais.
- A variável target é a que tem maior amplitude e um alto desvio padrão.

In [551]:
df_costs.describe()

,age,bmi,children,medical charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


### Transformar coluna smoker em numérica

In [552]:
df_costs['smoker'] = df_costs['smoker'].map({ 'yes': 1, 'no': 0})

In [553]:
df_costs.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              1338 non-null   int64  
 1   sex              1338 non-null   str    
 2   bmi              1338 non-null   float64
 3   children         1338 non-null   int64  
 4   smoker           1338 non-null   int64  
 5   region           1338 non-null   str    
 6   medical charges  1338 non-null   float64
dtypes: float64(2), int64(3), str(2)
memory usage: 73.3 KB


## Análise Exploratória de Dados - EDA

### Distribuição de Custos Médicos

In [554]:
fig = px.histogram(
  data_frame=df_costs,
  x='medical charges',
  title="Distribuição de Custos Médicos",
  nbins=50,
)
fig.show()

### Distribuição de Idade
- Há muitas pessoas jovens, o que em geral, causa um gasto menor com saúde.
- Apesar do foco em pessoas muito jovens(18-19), o restante da distribuição é uniforme

In [555]:
fig = px.histogram(
  data_frame=df_costs,
  x='age',
  title="Distribuição de Idade",
)
fig.show()

### Distribuição Quantidade de Filhos
- Maioria não tem filho ou tem 1 filho.
- A distribuição tem cauda à direita.

In [556]:
fig = px.histogram(
  data_frame=df_costs,
  x='children',
  title="Distribuição de Quantidade de Filhos",
)
fig.show()

### Mostrar distribuição de IMC
- Assemelha-se a uma curva de distribuição normal.

In [557]:
fig = px.histogram(
  data_frame=df_costs,
  x='bmi',
  title="Distribuição de IMC",
)
fig.show()

### Distribuição Variável Gênero
- Base bem balanceada por gênero 

In [558]:
px.bar(
  df_costs.value_counts('sex'),
  title='Distribuição de Gênero'
)

### Distribuição Fumante
- Maioria da base não fuma.

In [559]:
px.bar(
  df_costs.value_counts('smoker'),
  title='Distribuição Fumantes'
)

### Distribuição Região
- Praticamente uniforme

In [560]:
fig = px.bar(
  df_costs.value_counts('region'),
  title='Distribuição por Região',
)
fig.show()

### Boxplot: Custos Médicos x Idade
- Vê-se que quanto mais velha a pessoa é, o gasto mínimo vai aumentando.

In [561]:
px.box(
  df_costs,
  x='age',
  y='medical charges',
  title='Boxplot de Custos Médicos por Idade',
)

### Boxplot: Custos Médicos por Gênero
- Homens tendem a gastar mais que mulheres, embora a mediana seja muito próxima.

In [562]:
px.box(
  df_costs,
  x='sex',
  y='medical charges',
  title='Boxplot de Custos Médicos por Gênero',
  color='sex',
  color_discrete_map={ 'female': 'darkred', 'male': 'darkblue'}
)

#### Teste t-Student - Gênero x Gasto com Saúde
**Teste de t-Student**:
- H0: Evidência de que as 2 populações não têm uma diferença prática na média. Qualquer variação é ruído.
- H1: Evidência de que as 2 populações têm uma diferença mensurável. Existe diferença entre os 2 grupos.
- p-value \>= 0.05: Não rejeita H0
- p-value \< 0.05: Rejeita H0 e aceita H1.

**Conclusão:** a diferença entre os dois grupos pode ser tomada como mensurável, logo essa variável precisa ser considerada pelo modelo.

In [563]:
result = ttest_ind(
  a=df_costs[df_costs['sex'] == 'male']['medical charges'],
  b=df_costs[df_costs['sex'] == 'female']['medical charges'],
  equal_var=False,
)

print(f"t-Student p-value: {result.pvalue}") # type: ignore
if result.pvalue < 0.05: # type: ignore
    print('Rejeita H0 e aceita H1. Há evidência de diferença.')
else:
    print("Não rejeita H0. Não há evidência de diferença.")

t-Student p-value: 0.03584101495601666
Rejeita H0 e aceita H1. Há evidência de diferença.


### Boxplot: Custos Médicos x Fumante
- Os fumantes gastam mais.

In [564]:
px.box(
  df_costs,
  x='smoker',
  y='medical charges',
  title='Boxplot de Custos Médicos com influência de Fumar ou Não.',
  color='smoker',
  color_discrete_map={ 0: 'blue', 1:'red'}
)

### Boxplot: Custos Médicos por Região
- Mediana bem próxima, mas há variação de máximos.

In [565]:
px.box(
  df_costs,
  x='region',
  y='medical charges',
  title='Boxplot de Custos Médicos por Região',
  color='region',
)

### Correlação de Pearson

- Correlação forte entre `smoker` e `medical charges`

In [566]:
df_dummies = df_costs.copy()
df_dummies = pd.get_dummies(data=df_dummies, columns=['region', 'sex'])
corr_pearson = df_dummies.corr()

fig = go.Figure(
  go.Heatmap(
    x=corr_pearson.columns,
    y=corr_pearson.index,
    z=np.array(corr_pearson),
    text=corr_pearson.values,
    texttemplate='%{text:.3f}',
    colorscale=px.colors.diverging.RdBu_r,
    zmin=-1,
    zmax=1,
  )
)

fig.update_layout(title='Matriz de Correlação de Pearson')
fig.show()

## Preparação dos Dados

In [567]:
X = df_costs.drop(columns=['medical charges'])
y = df_costs['medical charges']

In [568]:
numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(include='str').columns.tolist()

preprocessor = ColumnTransformer(transformers=[
  ('num', StandardScaler(), numeric_features),
  ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

In [569]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

X_train_transformed = preprocessor.fit_transform(X=X_train)
X_test_transformed = preprocessor.transform(X=X_test)

print(X_train_transformed.shape)
print(X_test_transformed.shape)

(1070, 10)
(268, 10)


## Treinamento do Modelo

In [570]:
bagging_model = BaggingRegressor(
  estimator=LinearRegression(),
  n_estimators=10,
  # max_samples=0.5,
  # max_features=0.7,
  random_state=51,
)

In [571]:
bagging_model.fit(X_train_transformed, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeRegressor`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",LinearRegression()
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",51
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",10
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


## Análise de Métricas

In [572]:
y_pred = bagging_model.predict(X_test_transformed)
y_pred

array([ 8867.59078872, 36805.27813018,  2786.29194648, 11178.02788322,
       34023.55812248, 11612.68697707, 11555.24008631, 14975.75609284,
        5348.73841345, 10644.35007315,  9542.26478237, 12182.41562307,
        9963.18077502,  4197.02159732,  5495.5510617 , 12669.2259207 ,
        5654.27949942,  4898.19412627, 25738.02597205, 28755.29531981,
       10316.69436656,  8508.57070827, 32483.07329171, 13179.23439342,
        6165.46369588, 16089.92921178,  9917.10593579,  2575.88579687,
       23361.22140021,  8252.34064   ,  3894.4856253 , 30294.22573616,
        5737.14627997,  4710.06736037,  7789.91871347, 11150.60813685,
       13288.03609118,  2140.55532123, 12153.81159574,  7783.63881065,
        9844.16523243,   822.46724682,  5950.84135369,  2147.76049849,
        4277.11067328, 15121.58202627, 15339.42995097, 35018.97097488,
        8205.65604559, 12731.46150312,  5643.31124535, 30716.57617381,
        6984.39166773, 39973.66994033,  4436.59198834, 27572.45121014,
      

### Visualizar métricas
- Considerando RMSE e MAE, percebe-se que os outliers influenciam bastante no erro.
- Ter um erro de ≃4500 dólares é algo muito alto.

In [573]:
mae = mean_absolute_error(y_pred=y_pred, y_true=y_test)
rmse = root_mean_squared_error(y_pred=y_pred, y_true=y_test)
r2 = r2_score(y_pred=y_pred, y_true=y_test)

In [574]:
print(f"Mean Absolute Error: {mae:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f}")
print(f"R2-Score: {r2:.4f}")

Mean Absolute Error: 4542.7915
Root Mean Squared Error: 6613.2088
R2-Score: 0.7485


## Análise de Importância das Features

### Obter importância percentual das features

In [575]:
coefs = np.array([estimator.coef_ for estimator in bagging_model.estimators_])
features_importance = np.mean(np.absolute(coefs), axis=0)
features_importance_percentage = features_importance / np.sum(features_importance)
features_importance_percentage

array([0.18826013, 0.1133771 , 0.0296979 , 0.52434395, 0.0141138 ,
       0.0141138 , 0.03750999, 0.02108574, 0.03199751, 0.02550008])

### Obter nomes das features

In [576]:
feature_names = preprocessor.get_feature_names_out()
feature_names

array(['num__age', 'num__bmi', 'num__children', 'num__smoker',
       'cat__sex_female', 'cat__sex_male', 'cat__region_northeast',
       'cat__region_northwest', 'cat__region_southeast',
       'cat__region_southwest'], dtype=object)

### Barplot: Importância por Feature

In [577]:
df_importance = pd.DataFrame({ 'feature': feature_names, 'importance': features_importance_percentage})
df_importance.sort_values(by='importance', ascending=True, inplace=True)
px.bar(
  data_frame=df_importance,
  orientation='h',
  x='importance',
  y='feature',
)

## Propriedades do Modelo

### Max Samples
- No modelo original, todos os estimadores treinam com os mesmos dados e usando todos os dados.
- O parâmetro `max_samples` faz com que cada sample tenha somente xx% dos dados de forma aleatória para cada uma.

In [578]:
bagging_model.estimators_samples_[0].shape

(1070,)

### Max Features
- No modelo original, todos os estimadores treinam com todas as colunas.
- O parâmetro `max_features` faz com que cada sample tenha somente xx% das colunas de forma aleatória para cada uma.

In [579]:
bagging_model.estimators_features_

[array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])]

## Salvar Dados e Preprocessor

### Preprocessor

In [580]:
import joblib

joblib.dump(preprocessor, 'preprocessor_regressor.pkl')

['preprocessor_regressor.pkl']

### Dados

In [581]:
df_costs.to_csv('./dataset/cleaned_data.csv', index=False)